# CLEIDS-Edge — Notebook 06: CPU-Only Edge Latency/Throughput Benchmark

Measures per-sample inference latency, throughput, and real model size — **CPU-only, single-thread,
no exceptions** — for CLEIDS-Edge (original checkpoints AND the 16x8-quantized TFLite artifacts from
Notebook 05) and all 7 baselines from Notebook 04. This is the number set that supports the
"edge-deployable" claim (Contribution 1), so measuring it honestly on CPU with threading forced to 1
matters more here than in any other notebook.

**CPU-only, single-thread enforcement**: `tf.config.set_visible_devices([], "GPU")` plus
`tf.config.threading.set_{intra,inter}_op_parallelism_threads(1)`, and `OMP_NUM_THREADS`/
`MKL_NUM_THREADS`/`OPENBLAS_NUM_THREADS` set to `"1"` before NumPy/scikit-learn are imported (BLAS
libraries read these once at first use, so they must be set first). This matches a real low-power
single-core IoT edge device far better than Colab's underlying multi-core CPU running unconstrained.

**Real problem found while planning this notebook, not assumed**: Random Forest and SVM were never
saved as model artifacts in Notebook 04 — `train_and_evaluate_baseline()` fits them, evaluates them,
and lets them go out of scope; only their metrics were persisted to `baseline_results.json`. There is
nothing on disk to load for a genuine latency benchmark. Rather than fabricate numbers or skip these
two baselines, **both are retrained fresh here**, using the exact same `build_random_forest`/
`build_svm` functions, same hyperparameters, same `random_state=42`, and the same linear-SVM fallback
rule (train rows > 50,000) as Notebook 04 — deterministic training on the same real data means this
is a genuine re-derivation of the same model, not a different one. Random Forest is trained with
`n_jobs=-1` (practical training time) but **benchmarked with `n_jobs=1`** — training speed doesn't
need to match the edge deployment scenario, but the reported latency number must.

**Methodology decision, disclosed**: latency is measured at **batch size 1** (true single-sample
inference, the realistic edge scenario — an IoT device processes one flow/packet at a time, it does
not batch). Throughput is derived from that same measurement as `1000 / latency_ms_mean`, not from a
separate large-batch run — this keeps the methodology identical and directly comparable across Keras,
TFLite, and scikit-learn runtimes (a batch-512 TFLite artifact would need a second, differently-shaped
conversion, and a large-batch number would not represent genuine edge deployment anyway). Each
measurement discards the first 20 calls as warmup (JIT/cache effects that a real long-running service
would not pay per-inference) before timing 200 real calls; mean, std, and p95 (ms) are all recorded, not
just the mean.

**Peak memory was attempted and dropped — genuinely infeasible in this Colab environment (real finding,
verified via two independent methods, not given up on lightly).** The first attempt (per-model delta via
`resource.getrusage().ru_maxrss` inside the shared benchmark loop) turned out broken: `ru_maxrss` is a
cumulative, process-wide high-water mark that never resets, so across 55 sequential benchmarks sharing
one process only the very first model ever registered a nonzero delta. The real fix — running each model
in its own isolated subprocess (`scripts/measure_peak_memory_worker.py`) so `ru_maxrss` genuinely
reflects one model — was built, verified locally (four different model kinds gave real, distinct
values), and run on Colab. There, every single one of the 55 entries came back with the *exact same*
value, ~5.36GB, regardless of model type or size. A bare `python3 -c "import resource; print(...)"` with
nothing else loaded showed the identical number, and reading `/proc/self/status` directly (a completely
different OS mechanism) showed the same value again. Two independent measurement techniques, both
reporting an identical constant regardless of what's actually running — that is conclusive evidence this
specific Colab sandbox does not expose real per-process memory accounting, not a bug in this project's
code. **Peak memory is therefore not reported by this notebook.** Model size (measured for all 55
entries, real and correct) is used as the practical memory-footprint proxy in the thesis instead — for
these architectures, on-device memory scales closely with parameter/weight storage size.

**Pruning is deliberately NOT latency-benchmarked here.** Notebook 05 (§3f) already established that
one-shot magnitude pruning zeroes weights but stores them densely — without a sparse-aware runtime
(which TFLite's standard interpreter does not provide), a pruned model runs at the *same* latency as the
unpruned one; only its gzip-compressed storage size differs, which Notebook 05 already measured
correctly. Benchmarking pruned-model latency here would produce a real number that is trivially
uninformative (identical to the unquantized original) and could misleadingly imply a speed benefit that
does not exist. The quantized (16x8) variant genuinely is faster/smaller on real hardware and is
benchmarked below.

**No fabricated numbers** — every measurement comes from a real timed loop against this project's
actual test data and actual trained models (freshly retrained for RF/SVM, loaded from disk for
everything else), same discipline as every other notebook.

**Incremental backup**: save + Drive backup + GitHub push happen after every dataset within each model
block, not deferred to the end — the lesson from Notebook 04's real data loss (project brief §3b).

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"[DEBUG] userdata.get('GITHUB_TOKEN') raised {type(e).__name__}: {e}")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar) "
        "if running in the real Colab UI, or set os.environ['GITHUB_TOKEN'] manually for "
        "this session if running over a proxied connection."
    )

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)
subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "obololastkiller@gmail.com"])
subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Bright Adu-Boahene"])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Google Drive mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, "data_processed")
for d in (DRIVE_MODELS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT)


## 3. Data bridge -- copy processed splits from Google Drive

In [ ]:
import shutil

DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot", "iot-23"]

if not os.path.isdir(DRIVE_DATA_PROCESSED):
    raise RuntimeError(
        f"{DRIVE_DATA_PROCESSED} not found. Upload data/processed/ to Google Drive at "
        f"MyDrive/CLEIDS_Edge/data_processed/ first (same data Notebooks 03/04/05 used)."
    )

# train.npz is needed here (unlike Notebook 05) to genuinely retrain Random Forest and SVM --
# see the intro note on why no RF/SVM checkpoint exists to just load.
for name in DATASETS:
    src_dir = os.path.join(DRIVE_DATA_PROCESSED, name)
    dst_dir = f"data/processed/{name}"
    os.makedirs(dst_dir, exist_ok=True)
    for fn in ["train.npz", "val.npz", "test.npz", "label_classes.json", "feature_names.json"]:
        src = os.path.join(src_dir, fn)
        dst = os.path.join(dst_dir, fn)
        if not os.path.exists(src):
            raise RuntimeError(f"{src} not found on Drive. Re-upload data/processed/{name}/ and re-run.")
        if os.path.exists(dst):
            print(f"[skip] {dst} already present")
            continue
        shutil.copy2(src, dst)
        print(f"[copied] {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

manifest_dst = "data/processed/preprocessing_manifest.json"
if not os.path.exists(manifest_dst):
    shutil.copy2(os.path.join(DRIVE_DATA_PROCESSED, "preprocessing_manifest.json"), manifest_dst)

print("\nAll processed train+val+test split data copied from Drive.")


## 4. Setup -- CPU-only, single-thread enforced (do this before importing NumPy/sklearn)

In [ ]:
import os

# Must be set before NumPy / scikit-learn are imported -- their BLAS backends
# (OpenBLAS/MKL) read thread-count env vars once, at first use.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import time
import json
import shutil
import subprocess
import pickle
import resource
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers as klayers

tf.config.set_visible_devices([], "GPU")
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU visible after disabling:", tf.config.list_physical_devices("GPU"), "(must be [] -- CPU-only benchmark)")
print("TF intra-op threads:", tf.config.threading.get_intra_op_parallelism_threads())
print("TF inter-op threads:", tf.config.threading.get_inter_op_parallelism_threads())

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import build_random_forest, build_svm

for d in ["results"]:
    os.makedirs(d, exist_ok=True)

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)

SVM_LINEAR_FALLBACK = {}
for name in DATASETS:
    n_train = prep_manifest["datasets"][name]["shapes"]["train"][0]
    SVM_LINEAR_FALLBACK[name] = n_train > 50_000
print("SVM linear-fallback per dataset (train rows > 50,000):", SVM_LINEAR_FALLBACK)

NUM_CLASSES = {}
for name in DATASETS:
    with open(f"data/processed/{name}/label_classes.json") as f:
        NUM_CLASSES[name] = len(json.load(f)["classes"])
print("Num classes per dataset:", NUM_CLASSES)


## 5. Utilities

`build_export_model` / `make_representative_dataset` / `quantize_to_tflite` are the same real,
verified 16x8-quantization pipeline from Notebook 05 (project brief SS3e) -- reused as-is, but with
`batch_size=1` here (fixed single-sample shape) instead of Notebook 05's `batch_size=512`, since this
notebook measures true single-sample edge latency, not batched accuracy evaluation.

In [ ]:
def build_export_model(input_dim, num_classes, binary, batch_size):
    """TFLite-export-only rebuild: fixed batch_shape + LSTM(unroll=True). Same
    architecture as build_cleids_edge, weights transferred via set_weights --
    never used for CLEIDS-Edge's real (.keras) latency numbers, only to produce
    the quantized TFLite artifact."""
    inputs = tf.keras.Input(batch_shape=(batch_size, input_dim, 1), name="input")
    x = klayers.Conv1D(64, kernel_size=3, activation="relu", name="conv1d_block1")(inputs)
    x = klayers.BatchNormalization(name="bn1")(x)
    x = klayers.MaxPooling1D(pool_size=2, name="pool1")(x)
    x = klayers.Conv1D(128, kernel_size=3, activation="relu", name="conv1d_block2")(x)
    x = klayers.BatchNormalization(name="bn2")(x)
    x = klayers.MaxPooling1D(pool_size=2, name="pool2")(x)
    x = klayers.Dropout(0.3, name="dropout_conv")(x)
    x = klayers.LSTM(100, return_sequences=False, unroll=True, name="lstm")(x)
    x = klayers.Dropout(0.3, name="dropout_lstm")(x)
    x = klayers.Dense(64, activation="relu", name="dense_1")(x)
    if binary:
        outputs = klayers.Dense(1, activation="sigmoid", name="output")(x)
    else:
        outputs = klayers.Dense(num_classes, activation="softmax", name="output")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="CLEIDS_Edge_export")


def make_representative_dataset(X_val, batch_size=1, n_calib=1024):
    """Real calibration data for 16x8 quantization, sampled from the VALIDATION
    split (never test) -- test data stays untouched, same discipline as Notebook 05.
    n_calib reduced from Notebook 05's 2048 since batch_size=1 here means one
    forward pass per calibration sample regardless (more, smaller steps) --
    1024 real samples is still a solid calibration set."""
    n_calib = min(n_calib, X_val.shape[0])
    calib = X_val[:n_calib]
    n_padded = int(np.ceil(n_calib / batch_size)) * batch_size
    calib_padded = np.zeros((n_padded,) + X_val.shape[1:], dtype=np.float32)
    calib_padded[:n_calib] = calib

    def representative_dataset():
        for start in range(0, n_padded, batch_size):
            yield [calib_padded[start:start + batch_size]]

    return representative_dataset


def quantize_to_tflite(orig_model, input_dim, num_classes, binary, tflite_path, X_val, batch_size=1):
    """16x8 quantization (INT16 activations, INT8 weights) -- the real, verified
    fix from Notebook 05 (project brief SS3e); pure INT8 dynamic-range/full-INT8
    quantization crashes or collapses accuracy on this architecture's unrolled
    LSTM. See SS3e for the full diagnostic record."""
    export_model = build_export_model(input_dim, num_classes, binary, batch_size)
    export_model.set_weights(orig_model.get_weights())
    converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset(X_val, batch_size)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8
    ]
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32
    tflite_bytes = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_bytes)
    return len(tflite_bytes)


def time_latency_keras(model, X, n_warmup=20, n_repeat=200, seed=42):
    """Single-sample (batch=1) latency, ms. First n_warmup calls discarded
    (one-time JIT/cache setup cost a real long-running service would not pay
    per inference)."""
    rng = np.random.RandomState(seed)
    n_total = n_warmup + n_repeat
    idx = rng.choice(X.shape[0], size=n_total, replace=(X.shape[0] < n_total))
    times = []
    for i, ix in enumerate(idx):
        sample = X[ix:ix + 1]
        t0 = time.perf_counter()
        model.predict(sample, batch_size=1, verbose=0)
        t1 = time.perf_counter()
        if i >= n_warmup:
            times.append((t1 - t0) * 1000)
    times = np.array(times)
    return {
        "latency_ms_mean": float(times.mean()), "latency_ms_std": float(times.std()),
        "latency_ms_p95": float(np.percentile(times, 95)),
    }


def time_latency_tflite(tflite_path, X, n_warmup=20, n_repeat=200, seed=42):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.allocate_tensors()
    rng = np.random.RandomState(seed)
    n_total = n_warmup + n_repeat
    idx = rng.choice(X.shape[0], size=n_total, replace=(X.shape[0] < n_total))
    times = []
    for i, ix in enumerate(idx):
        sample = X[ix:ix + 1].astype(np.float32)
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]["index"])
        t1 = time.perf_counter()
        if i >= n_warmup:
            times.append((t1 - t0) * 1000)
    times = np.array(times)
    return {
        "latency_ms_mean": float(times.mean()), "latency_ms_std": float(times.std()),
        "latency_ms_p95": float(np.percentile(times, 95)),
    }


def time_latency_sklearn(model, X, n_warmup=20, n_repeat=200, seed=42):
    rng = np.random.RandomState(seed)
    n_total = n_warmup + n_repeat
    idx = rng.choice(X.shape[0], size=n_total, replace=(X.shape[0] < n_total))
    times = []
    for i, ix in enumerate(idx):
        sample = X[ix:ix + 1]
        t0 = time.perf_counter()
        model.predict_proba(sample)
        t1 = time.perf_counter()
        if i >= n_warmup:
            times.append((t1 - t0) * 1000)
    times = np.array(times)
    return {
        "latency_ms_mean": float(times.mean()), "latency_ms_std": float(times.std()),
        "latency_ms_p95": float(np.percentile(times, 95)),
    }


def finalize_latency_result(latency_dict, size_mb):
    return {
        **latency_dict,
        "throughput_samples_per_sec": round(1000.0 / latency_dict["latency_ms_mean"], 2),
        "size_mb": round(size_mb, 4),
    }


def save_latency_results(new_results):
    """Additive load-then-update, called after every dataset -- same
    discipline as Notebooks 04/05 after their real data-loss incidents."""
    path = "results/latency_results.json"
    existing = {}
    if os.path.exists(path):
        with open(path) as f:
            existing = json.load(f)
    existing.update(new_results)
    with open(path, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"Wrote {path} ({len(existing)} entries total)")
    return existing


def backup_and_push(commit_message):
    shutil.copy2("results/latency_results.json", os.path.join(DRIVE_RESULTS, "latency_results.json"))
    subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "results/"], check=False)
    commit_res = subprocess.run(
        ["git", "-C", REPO_DIR, "commit", "-m", commit_message], capture_output=True, text=True,
    )
    print(commit_res.stdout, commit_res.stderr)
    if commit_res.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
        print(f"Pushed: {commit_message}")
    else:
        print("Nothing new to commit (or commit failed) -- see output above.")


## 6. Benchmark: CLEIDS-Edge (original .keras + 16x8-quantized TFLite)

In [ ]:
for dataset_name in DATASETS:
    data_dir = f"data/processed/{dataset_name}"
    test_data = np.load(f"{data_dir}/test.npz")
    val_data = np.load(f"{data_dir}/val.npz")
    X_test = test_data["X_cnn"].astype(np.float32)
    X_val = val_data["X_cnn"].astype(np.float32)
    input_dim = X_test.shape[1]
    num_classes = NUM_CLASSES[dataset_name]

    for task, binary in [("binary", True), ("multiclass", False)]:
        ckpt_name = f"cleids_edge_{dataset_name}_{task}"
        ckpt_path = f"models/{ckpt_name}.keras"
        print("\n" + "=" * 70)
        print(f"[BENCHMARKING] {ckpt_name}")
        print("=" * 70)

        model = tf.keras.models.load_model(ckpt_path)

        lat = time_latency_keras(model, X_test)
        size_mb = os.path.getsize(ckpt_path) / 1e6
        result_orig = finalize_latency_result(lat, size_mb)
        print(f"[ORIGINAL]  latency={result_orig['latency_ms_mean']:.3f}ms (p95={result_orig['latency_ms_p95']:.3f}) "
              f"throughput={result_orig['throughput_samples_per_sec']:.1f}/s size={result_orig['size_mb']:.3f}MB")

        tflite_path = f"models/{ckpt_name}_latbench.tflite"
        quantize_to_tflite(model, input_dim, num_classes, binary, tflite_path, X_val, batch_size=1)
        lat_q = time_latency_tflite(tflite_path, X_test)
        size_q = os.path.getsize(tflite_path) / 1e6
        result_quant = finalize_latency_result(lat_q, size_q)
        print(f"[QUANTIZED] latency={result_quant['latency_ms_mean']:.3f}ms (p95={result_quant['latency_ms_p95']:.3f}) "
              f"throughput={result_quant['throughput_samples_per_sec']:.1f}/s size={result_quant['size_mb']:.3f}MB")
        os.remove(tflite_path)

        save_latency_results({
            f"{ckpt_name}__original": result_orig,
            f"{ckpt_name}__quantized": result_quant,
        })

    backup_and_push(f"Notebook 06: CLEIDS-Edge latency benchmark for {dataset_name}")

print("\nCLEIDS-Edge latency benchmark complete (original + quantized, all 5 datasets x binary/multiclass).")


## 7. Benchmark: Keras-based baselines (5 architectures x 5 datasets, binary only)

In [ ]:
KERAS_BASELINES = ["standalone_cnn", "standalone_lstm", "nazir2024", "altaie_hoomod2024", "wang2023_dlbilstm"]

for model_name in KERAS_BASELINES:
    for dataset_name in DATASETS:
        data_dir = f"data/processed/{dataset_name}"
        test_data = np.load(f"{data_dir}/test.npz")
        X_test = test_data["X_cnn"].astype(np.float32)
        ckpt_path = f"models/{model_name}_{dataset_name}_binary.keras"
        if not os.path.exists(ckpt_path):
            print(f"[SKIP] {ckpt_path} not found -- not benchmarked, not fabricated.")
            continue

        print("\n" + "=" * 70)
        print(f"[BENCHMARKING] {model_name}/{dataset_name}")
        print("=" * 70)

        model = tf.keras.models.load_model(ckpt_path)
        lat = time_latency_keras(model, X_test)
        size_mb = os.path.getsize(ckpt_path) / 1e6
        result = finalize_latency_result(lat, size_mb)
        print(f"latency={result['latency_ms_mean']:.3f}ms (p95={result['latency_ms_p95']:.3f}) "
              f"throughput={result['throughput_samples_per_sec']:.1f}/s size={result['size_mb']:.3f}MB")

        save_latency_results({f"{model_name}_{dataset_name}_binary": result})

    backup_and_push(f"Notebook 06: {model_name} latency benchmark (all datasets)")

print("\nKeras-based baseline latency benchmark complete.")


## 8. Benchmark: Random Forest + SVM (retrained fresh -- see intro for why)

No checkpoint exists for these two (Notebook 04 never saved them). Retrained here with the identical
`build_random_forest`/`build_svm` functions, hyperparameters, and `random_state=42` used in Notebook 04
-- deterministic training on the same real data, a genuine re-derivation, not a different model.

In [ ]:
for dataset_name in DATASETS:
    data_dir = f"data/processed/{dataset_name}"
    train_data = np.load(f"{data_dir}/train.npz")
    test_data = np.load(f"{data_dir}/test.npz")
    X_train = train_data["X_flat"]
    y_train = train_data["y_bin"].astype(int)
    X_test = test_data["X_flat"].astype(np.float32)

    print("\n" + "=" * 70)
    print(f"[RETRAINING + BENCHMARKING] random_forest/{dataset_name}")
    print("=" * 70)
    rf = build_random_forest()
    t0 = time.time()
    rf.fit(X_train, y_train)
    print(f"  fit took {time.time()-t0:.1f}s (n_jobs=-1 for training speed; benchmarked at n_jobs=1 below)")
    rf.set_params(n_jobs=1)
    lat = time_latency_sklearn(rf, X_test)
    size_mb = len(pickle.dumps(rf)) / 1e6
    result_rf = finalize_latency_result(lat, size_mb)
    print(f"latency={result_rf['latency_ms_mean']:.3f}ms (p95={result_rf['latency_ms_p95']:.3f}) "
          f"throughput={result_rf['throughput_samples_per_sec']:.1f}/s size={result_rf['size_mb']:.3f}MB")

    print("\n" + "=" * 70)
    print(f"[RETRAINING + BENCHMARKING] svm/{dataset_name}")
    print("=" * 70)
    svm_variant = "linear (fallback)" if SVM_LINEAR_FALLBACK[dataset_name] else "kernel (rbf)"
    svm = build_svm(use_linear_fallback=SVM_LINEAR_FALLBACK[dataset_name])
    t0 = time.time()
    svm.fit(X_train, y_train)
    print(f"  fit took {time.time()-t0:.1f}s ({svm_variant})")
    lat = time_latency_sklearn(svm, X_test)
    size_mb = len(pickle.dumps(svm)) / 1e6
    result_svm = finalize_latency_result(lat, size_mb)
    print(f"latency={result_svm['latency_ms_mean']:.3f}ms (p95={result_svm['latency_ms_p95']:.3f}) "
          f"throughput={result_svm['throughput_samples_per_sec']:.1f}/s size={result_svm['size_mb']:.3f}MB")

    save_latency_results({
        f"random_forest_{dataset_name}_binary": result_rf,
        f"svm_{dataset_name}_binary": result_svm,
    })
    backup_and_push(f"Notebook 06: RF+SVM latency benchmark for {dataset_name}")

print("\nRandom Forest + SVM latency benchmark complete (retrained fresh, all 5 datasets).")


## 9. Peak-memory: attempted, found genuinely infeasible, dropped (real finding, not skipped lightly)

**First attempt** (Sections 6-8 above): per-model delta via `resource.getrusage().ru_maxrss` inside the
shared benchmark loop. Broken: `ru_maxrss` is a cumulative, process-wide high-water mark that never
resets, so across 55 sequential benchmarks sharing one process, only the very first model measured
(`cleids_edge_nsl-kdd_binary`) ever registered a nonzero delta -- every one of the other 54 entries,
including Random Forest checkpoints 150-287MB on disk, read exactly `0.00MB`. Not a real finding about
those models' memory use -- an artifact of measuring a shared, long-running process.

**Second attempt**: `scripts/measure_peak_memory_worker.py` ran each model in its own fresh, isolated
Python subprocess, so `ru_maxrss` at the end of a dedicated single-model process should genuinely be
that model's real peak -- the standard, correct way to do per-model memory profiling, and verified
working locally (four different model kinds gave real, distinct values: 422.8MB, 492.4MB, 172.3MB,
101.4MB). On Colab, this also broke: all 55 entries came back with the *exact same* value, ~5.36GB,
regardless of model type. Confirmed via two independent diagnostics that this is a genuine Colab sandbox
limitation, not a bug in this project's code: a bare `python3 -c "import resource; print(...)"` with
nothing else loaded showed the identical ~5.36GB figure, and reading `/proc/self/status` directly (a
completely different OS mechanism) from the notebook's own kernel process showed the same number again.
Two independent measurement techniques, both reporting an identical constant regardless of what's
actually running -- conclusive evidence this sandbox does not expose real per-process memory accounting.

**Decision**: peak memory is dropped as a directly-measured metric. Model size (real, correct, measured
for all 55 entries in Sections 6-8) is used as the practical memory-footprint proxy in the thesis
instead. The cell below is now just cleanup -- it strips any stray `peak_memory_mb` field left over from
the two abandoned attempts, so the final `results/latency_results.json` doesn't carry misleading
half-measured values forward.

In [ ]:
with open("results/latency_results.json") as f:
    latency_results = json.load(f)

stripped = 0
for key, r in latency_results.items():
    if "peak_memory_mb" in r:
        del r["peak_memory_mb"]
        stripped += 1

with open("results/latency_results.json", "w") as f:
    json.dump(latency_results, f, indent=2)

print(f"Stripped stray peak_memory_mb from {stripped}/{len(latency_results)} entries.")
backup_and_push("Notebook 06: drop peak_memory_mb (genuinely infeasible in this Colab sandbox, see section 9)")


## 10. Final summary table

In [ ]:
with open("results/latency_results.json") as f:
    latency_results = json.load(f)

print("=" * 90)
print("CLEIDS-Edge -- Notebook 06 Headline Summary (CPU-only, single-thread, batch=1)")
print("=" * 90)
print(f"{'entry':40s} {'latency_ms':>11s} {'throughput/s':>13s} {'size_mb':>9s}")
print("-" * 90)
for name in sorted(latency_results.keys()):
    r = latency_results[name]
    print(f"{name:40s} {r['latency_ms_mean']:11.3f} {r['throughput_samples_per_sec']:13.1f} "
          f"{r['size_mb']:9.4f}")
print("=" * 90)
print(f"Total entries: {len(latency_results)}")
print("Full detail (std, p95) in results/latency_results.json.")
print("Peak memory is not reported -- genuinely infeasible in this Colab sandbox, see section 9.")
print("Model size is used as the memory-footprint proxy in the thesis instead.")


## 11. Final backup + push

In [ ]:
shutil.copy2("results/latency_results.json", os.path.join(DRIVE_RESULTS, "latency_results.json"))
subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "results/", "notebooks/06_Edge_Latency_Benchmark_CPU.ipynb"], check=False)
commit_res = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 06: final CPU-only latency/throughput benchmark (all models)"],
    capture_output=True, text=True,
)
print(commit_res.stdout, commit_res.stderr)
if commit_res.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed final results.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")
